[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/34_speculative_decoding.ipynb)

# 🔴 Hard: Speculative Decoding

Implement the **acceptance/rejection step** of speculative decoding — a technique for accelerating LLM inference.

### Signature
```python
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]:
    # target_probs: (K, V) from target (large) model
    # draft_probs: (K, V) from draft (small) model
    # draft_tokens: (K,) tokens sampled by draft model
    # Returns: list of accepted tokens (1 to K)
```

### Algorithm
For each position i = 0, ..., K-1:
1. `ratio = target_probs[i, token_i] / draft_probs[i, token_i]`
2. Accept with probability `min(1, ratio)`
3. If rejected: sample from `normalize(max(0, target - draft))`, append, and stop

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import torch

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

def speculative_decode(target_probs, draft_probs, draft_tokens):
  K,V=target_probs.shape
  accepted_tokens=[]
  for i in range(K):
    t_p=target_probs[i,draft_tokens[i]]
    d_p=draft_probs[i,draft_tokens[i]]

    accept_prob=min(1.0,t_p/d_p)

    if torch.rand(1).item()<accept_prob:
      accepted_tokens.append(draft_tokens[i].item())
    else:
      diff = torch.clamp(target_probs[i] - draft_probs[i], min=0)
      next_token = torch.multinomial(diff / diff.sum(), 1).item()
      accepted_tokens.append(next_token)
  return accepted_tokens


In [12]:
# 🧪 Debug
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('Perfect draft:', speculative_decode(probs, probs, tokens))
target = torch.softmax(torch.randn(4, 10), dim=-1)
draft = torch.softmax(torch.randn(4, 10), dim=-1)
print('Random draft:', speculative_decode(target, draft, tokens))

Perfect draft: [2, 5, 1, 8]
Random draft: [2, 5, 1, 8]


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('speculative_decoding')


🧪 Testing: Speculative Decoding (Hard)
──────────────────────────────────────────────────
  ✅ [1/3] Perfect draft: all accepted (2.0ms)
  ✅ [2/3] Output length bounded (46.3ms)
  ✅ [3/3] All tokens valid (17.7ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (66.0ms total)
  Progress saved. Run status() to see your dashboard.

